# Paso 1 -- Verificar las 8 fórmulas de recompensa en un paso de ejemplo

Antes de entrenar nada: confirmar que las 8 fórmulas (A-H -- ver
`modelo_rl/prueba_rewards/README.md`) se calculan bien y dan números
coherentes con lo esperado de cada una. A-D ya estaban verificadas; E-H son
nuevas (agregadas sin tocar A-D). Chequeos de sanity matemático incluidos:
con `eta_potencial=0`, C debe coincidir EXACTAMENTE con B; y para G (tiempo
incremental), la suma de `deltaT` a lo largo de un episodio completo debe
coincidir EXACTAMENTE con el tiempo total en sistema de cada persona
(atendida o activa al cierre) -- la identidad que garantiza que G no tiene
doble conteo ni fugas.

**Nota sobre C y G:** ambas necesitan "cómo estaba el mundo antes de este
paso". La primera implementación guardaba el objeto `EstadoSimulacion` de
un paso anterior para leerlo en el siguiente -- tenía un bug real:
`EstadoSimulacion.colas`/`.barcos` se pasan por REFERENCIA desde `env.py`
(`_construir_estado`), así que un estado "viejo" guardado no queda
congelado, sus colas/barcos son los mismos objetos que el simulador sigue
mutando después. La fórmula correcta usa un ESCALAR precalculado
(`potencial_antes` para C, `s_tiempo_antes` para G) en vez del objeto
completo -- ver el docstring de `entorno_recompensa_intercambiable.py` para
el detalle de cómo se descubrió y verificó.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

BASE_DIR = Path.cwd().parent
SIMULADOR_DIR = (BASE_DIR / "../../simulador").resolve()
POLITICA_BASE_DIR = (BASE_DIR / "../../politica_base").resolve()
MODELO_RL_DIR = (BASE_DIR / "..").resolve()
BB_DIR = (BASE_DIR / "../../bergen-boats").resolve()
DEMAND_DIR = (BASE_DIR / "../../demand").resolve()
sys.path.insert(0, str(DEMAND_DIR / "src"))
sys.path.insert(0, str(SIMULADOR_DIR / "src"))
sys.path.insert(0, str(POLITICA_BASE_DIR / "src"))
sys.path.insert(0, str(MODELO_RL_DIR / "src"))
sys.path.insert(0, str(BASE_DIR / "src"))

import masas as demand_masas
import llegadas as demand_llegadas
from politica_base import asignar_flota
from entrenamiento import EntornoDemandaAleatoria
from entorno_recompensa_intercambiable import EntornoRecompensaIntercambiable
from recompensas_alternativas import (
    calcular_recompensa_A, calcular_recompensa_B, calcular_recompensa_C, calcular_recompensa_D,
    calcular_recompensa_E, calcular_recompensa_F, calcular_recompensa_G, calcular_recompensa_H,
    potencial, suma_tiempo_activo,
)

cfg_propio = yaml.safe_load(open(BASE_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_sim = yaml.safe_load(open(SIMULADOR_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_bb = yaml.safe_load(open(BB_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_demand = demand_masas.cargar_config(DEMAND_DIR / "config" / "instance.yaml")

resumen_masas = pd.read_csv(DEMAND_DIR / "output" / "masas_por_nodo.csv", index_col="id")
intensidad_od = pd.read_csv(DEMAND_DIR / "output" / "matriz_intensidad_od.csv")
poblacion_total_zonas = resumen_masas["poblacion_total"].sum()
conexiones_fuertes = cfg_bb["garantia"]["conexiones_fuertes"]
nodos = [n["id"] for n in cfg_bb["nodos_demanda"]]
matriz_tiempos = pd.read_csv(BB_DIR / "02_ruteo_navegable" / "output" / "matriz_tiempos_min.csv", index_col=0)

cfg_recompensas = cfg_propio["recompensas"]
cfg_entren_propio = cfg_propio["entrenamiento"]
cfg_agente = cfg_sim["agente"]
cfg_hiper = cfg_agente["hiperparametros"]
cfg_escalon = cfg_sim["escalones"][cfg_entren_propio["escalon_base"]]
hora_ini_min, hora_fin_min = cfg_escalon["horas"][0] * 60, cfg_escalon["horas"][1] * 60
semillas_eval = cfg_agente["evaluacion"]["semillas"]
gamma = cfg_agente["gamma"]

print("Recompensas definidas:", list(cfg_recompensas.keys()))
for tipo, c in cfg_recompensas.items():
    nombre = c["nombre"]
    print(f"  {tipo}: {nombre}")
escalon_base = cfg_entren_propio["escalon_base"]
num_barcos, horas = cfg_escalon["num_barcos"], cfg_escalon["horas"]
print(f"\nInstancia: {escalon_base} ({num_barcos} barcos, {horas}h)")
timesteps = cfg_entren_propio["total_timesteps"]
print(f"Timesteps de entrenamiento (cada una): {timesteps}")
print(f"Semillas de evaluacion: {semillas_eval}")


def construir_entorno_base():
    return EntornoDemandaAleatoria(
        generar_llegadas_dia_fn=demand_llegadas.generar_llegadas_dia,
        cfg_demand=cfg_demand, intensidad_od=intensidad_od, conexiones_fuertes=conexiones_fuertes,
        poblacion_total_zonas=poblacion_total_zonas, horas=cfg_escalon["horas"],
        porcentaje_poblacion_dia=cfg_escalon["porcentaje_poblacion_dia"],
        matriz_tiempos=matriz_tiempos, nodos=nodos, num_barcos=cfg_escalon["num_barcos"],
        capacidad_barco=cfg_bb["flota"]["capacidad_pasajeros"], nodo_inicial=cfg_bb["flota"]["nodo_inicial"],
        paso_tiempo_min=cfg_sim["paso_tiempo_min"], hora_inicio_min=hora_ini_min, hora_fin_min=hora_fin_min,
        cfg_recompensa=cfg_sim["recompensa"], unidad_demanda=cfg_sim["unidad_demanda"],
    )


def construir_entorno_intercambiable(tipo_recompensa):
    return EntornoRecompensaIntercambiable(
        tipo_recompensa=tipo_recompensa,
        cfg_recompensa_tipo=cfg_recompensas[tipo_recompensa],
        gamma=gamma,
        generar_llegadas_dia_fn=demand_llegadas.generar_llegadas_dia,
        cfg_demand=cfg_demand, intensidad_od=intensidad_od, conexiones_fuertes=conexiones_fuertes,
        poblacion_total_zonas=poblacion_total_zonas, horas=cfg_escalon["horas"],
        porcentaje_poblacion_dia=cfg_escalon["porcentaje_poblacion_dia"],
        matriz_tiempos=matriz_tiempos, nodos=nodos, num_barcos=cfg_escalon["num_barcos"],
        capacidad_barco=cfg_bb["flota"]["capacidad_pasajeros"], nodo_inicial=cfg_bb["flota"]["nodo_inicial"],
        paso_tiempo_min=cfg_sim["paso_tiempo_min"], hora_inicio_min=hora_ini_min, hora_fin_min=hora_fin_min,
        cfg_recompensa=cfg_sim["recompensa"], unidad_demanda=cfg_sim["unidad_demanda"],
    )


Recompensas definidas: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
  A: Objetivo puro (identidad de Little)
  B: Ponderada convexa con tolerancia, sin techo
  C: B + potential-based shaping (Ng, Harada & Russell 1999)
  D: sin tolerancia  (parametros de produccion)
  E: Tiempo lineal puro (sin normalizar, sin movimiento)
  F: Tiempo cuadratico puro (sin normalizar, sin movimiento)
  G: Tiempo incremental (delta T por paso)
  H: Multiobjetivo 95% tiempo cuadratico + 5% movimiento

Instancia: escalon_1 (2 barcos, [6, 9]h)
Timesteps de entrenamiento (cada una): 150000
Semillas de evaluacion: [1001, 1002, 1003, 1004, 1005]


## Construir un par de estados (antes/después) no trivial

Se corre la política base unos pasos (semilla fija, reproducible) para que
haya gente esperando de verdad -- en el primer paso de una corrida las
colas están vacías y las 4 fórmulas darían 0, poco informativo.

In [2]:
env = construir_entorno_base()
obs, info = env.reset(seed=1001)

N_PASOS_PREVIOS = 15
for _ in range(N_PASOS_PREVIOS):
    estado = info["_estado_obj"]
    libres = [b for b in estado.barcos if b.libre]
    decisiones = asignar_flota(libres, estado, matriz_tiempos, env.capacidad_barco, cfg_sim["recompensa"])
    accion = np.array([env.codificar_accion_barco(decisiones[b.id]) if b.libre else 0 for b in estado.barcos])
    obs, r, terminated, truncated, info = env.step(accion)

# El paso de ejemplo: uno mas, con estado ANTES y DESPUES capturados aparte.
estado_antes = info["_estado_obj"]
libres = [b for b in estado_antes.barcos if b.libre]
decisiones = asignar_flota(libres, estado_antes, matriz_tiempos, env.capacidad_barco, cfg_sim["recompensa"])
accion = np.array([env.codificar_accion_barco(decisiones[b.id]) if b.libre else 0 for b in estado_antes.barcos])
len_atendidas_antes = len(env.atendidas_historico)
obs, r_original, terminated, truncated, info = env.step(accion)
estado_despues = info["_estado_obj"]
entregados_paso_ejemplo = env.atendidas_historico[len_atendidas_antes:]

n_antes = sum(u.tamano for cola in estado_antes.colas.values() for u in cola) + sum(len(b.a_bordo) for b in estado_antes.barcos)
n_despues = sum(u.tamano for cola in estado_despues.colas.values() for u in cola) + sum(len(b.a_bordo) for b in estado_despues.barcos)
print(f"Minuto del paso de ejemplo: {estado_despues.t_actual_min}")
print(f"Personas activas ANTES del paso: {n_antes}")
print(f"Personas activas DESPUES del paso: {n_despues}")
print(f"Entregadas EN este paso: {len(entregados_paso_ejemplo)}")


Minuto del paso de ejemplo: 392.0
Personas activas ANTES del paso: 27
Personas activas DESPUES del paso: 27
Entregadas EN este paso: 0


## Las 8 fórmulas, mismo paso, lado a lado

E-H son nuevas (agregadas después de A-D, sin tocarlas). Todas se calculan
sobre el mismo par `estado_antes`/`estado_despues` de arriba.

In [3]:
paso_tiempo_min = cfg_sim["paso_tiempo_min"]

potencial_antes = potencial(estado_antes, cfg_recompensas["C"]["eta_potencial"])
s_tiempo_antes = suma_tiempo_activo(estado_antes)

r_A, desglose_A = calcular_recompensa_A(estado_despues, cfg_recompensas["A"], paso_tiempo_min)
r_B, desglose_B = calcular_recompensa_B(estado_despues, cfg_recompensas["B"])
r_C, desglose_C = calcular_recompensa_C(estado_despues, potencial_antes, cfg_recompensas["C"], gamma)
r_D, desglose_D = calcular_recompensa_D(estado_despues, cfg_recompensas["D"])
r_E, desglose_E = calcular_recompensa_E(estado_despues, cfg_recompensas["E"])
r_F, desglose_F = calcular_recompensa_F(estado_despues, cfg_recompensas["F"])
r_G, desglose_G = calcular_recompensa_G(estado_despues, s_tiempo_antes, entregados_paso_ejemplo, cfg_recompensas["G"])
r_H, desglose_H = calcular_recompensa_H(estado_despues, cfg_recompensas["H"])

filas = []
for tipo, r, desglose in [
    ("A", r_A, desglose_A), ("B", r_B, desglose_B), ("C", r_C, desglose_C), ("D", r_D, desglose_D),
    ("E", r_E, desglose_E), ("F", r_F, desglose_F), ("G", r_G, desglose_G), ("H", r_H, desglose_H),
]:
    fila = {"tipo": tipo, "nombre": cfg_recompensas[tipo]["nombre"], "r": r}
    fila.update(desglose)
    filas.append(fila)

tabla_ejemplo = pd.DataFrame(filas)
display(tabla_ejemplo.round(4))
print("\nComo referencia, el reward que hubiera devuelto env.py con la formula de produccion (ignorado, se recalcula):", round(r_original, 4))


,tipo,nombre,r,tiempo_activo,movimiento,entrega,incomodidad,shaping,tiempo_lineal,tiempo_cuadratico,tiempo_incremental,tiempo_cuadratico_ponderado,movimiento_ponderado
0,A,Objetivo puro (identidad de Little),-54.0120,54.0,0.012,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,B,"Ponderada convexa con tolerancia, sin techo",-2.3155,NaN,0.006,0.0,2.3095,NaN,NaN,NaN,NaN,NaN,NaN
2,C,"B + potential-based shaping (Ng, Harada & Russ...",-2.3020,NaN,0.006,0.0,2.3095,0.0135,NaN,NaN,NaN,NaN,NaN
3,D,sin tolerancia (parametros de produccion),-48.0546,NaN,0.000,0.0,48.0546,NaN,NaN,NaN,NaN,NaN,NaN
4,E,"Tiempo lineal puro (sin normalizar, sin movimi...",-416.7900,NaN,0.000,0.0,NaN,NaN,416.79,NaN,NaN,NaN,NaN
5,F,"Tiempo cuadratico puro (sin normalizar, sin mo...",-6919.8579,NaN,0.000,0.0,NaN,NaN,NaN,6919.8579,NaN,NaN,NaN
6,G,Tiempo incremental (delta T por paso),-54.0000,NaN,0.000,0.0,NaN,NaN,NaN,NaN,54.0,NaN,NaN
7,H,Multiobjetivo 95% tiempo cuadratico + 5% movim...,-6573.9650,NaN,2.000,0.0,NaN,NaN,NaN,6919.8579,NaN,6573.865,0.1



Como referencia, el reward que hubiera devuelto env.py con la formula de produccion (ignorado, se recalcula): -2.3155


**Lectura esperada:** A es la más chica en magnitud por persona pero escala con
`tamaño x 2 min` de TODOS los activos (no solo los atrasados) así que puede
ser grande si hay mucha gente en el sistema, aunque nadie esté fuera de
tolerancia todavía. B y D usan la misma fórmula cuadrática, D con techo
bajo (1.0) y `peso_movimiento` mucho más caro (0.1 vs. 0.003 de B) -- si el
paso de ejemplo tiene algún barco moviéndose, D debería penalizar ese
movimiento mucho más fuerte que B. C es B más el término de shaping --
debería diferir de B en exactamente `desglose_C["shaping"]`.

## Chequeo de sanity: con `eta_potencial=0`, C debe coincidir EXACTAMENTE con B

`Phi(s) = -eta * N(s)` -- si `eta=0`, `Phi(s)=0` para CUALQUIER estado, así
que el término de shaping `gamma*Phi(s')-Phi(s)` es 0 sin importar qué tan
distintos sean `estado_antes` y `estado_despues`. No es un caso especial
de "estados iguales", es una propiedad de la fórmula: con `eta=0`, C debe
ser bit-a-bit igual a B, siempre.

In [4]:
cfg_C_eta_cero = dict(cfg_recompensas["C"], eta_potencial=0.0)
potencial_antes_eta_cero = potencial(estado_antes, 0.0)
r_C_eta_cero, desglose_C_eta_cero = calcular_recompensa_C(estado_despues, potencial_antes_eta_cero, cfg_C_eta_cero, gamma)

shaping_eta_cero = desglose_C_eta_cero["shaping"]
print(f"r_B             = {r_B!r}")
print(f"r_C (eta=0)     = {r_C_eta_cero!r}")
print(f"shaping (eta=0) = {shaping_eta_cero!r}")

assert desglose_C_eta_cero["shaping"] == 0.0, "con eta=0 el termino de shaping debe ser exactamente 0"
assert r_C_eta_cero == r_B, "con eta=0, C debe coincidir bit a bit con B"
print("\nOK: con eta_potencial=0, C == B exactamente.")


r_B             = -2.3155466049382696
r_C (eta=0)     = -2.3155466049382696
shaping (eta=0) = 0.0

OK: con eta_potencial=0, C == B exactamente.


## Segundo chequeo: el signo del shaping tiene sentido

Si la cola se achicó en este paso (`estado_despues` tiene menos gente
activa que `estado_antes`), `Phi` subió (menos negativo) y el término de
shaping debería ser POSITIVO -- un premio extra por progresar. Si la cola
creció, debería ser negativo.

In [5]:
eta = cfg_recompensas["C"]["eta_potencial"]
phi_antes = potencial(estado_antes, eta)
phi_despues = potencial(estado_despues, eta)
shaping = gamma * phi_despues - phi_antes

print(f"N(antes)={n_antes}, N(despues)={n_despues}")
print(f"Phi(antes)={phi_antes:.4f}, Phi(despues)={phi_despues:.4f}")
print(f"Termino de shaping = gamma*Phi(despues) - Phi(antes) = {shaping:.4f}")
if n_despues < n_antes:
    print("La cola se achico -> shaping deberia ser positivo:", "OK" if shaping > 0 else "revisar")
elif n_despues > n_antes:
    print("La cola crecio -> shaping deberia ser negativo:", "OK" if shaping < 0 else "revisar")
else:
    print("N no cambio -- shaping = (gamma-1)*Phi; con gamma<1 y Phi<0 (hay gente activa), (gamma-1)*Phi > 0: un premio chico y constante mientras la cola no se mueva, del orden de (1-gamma)*|Phi|")


N(antes)=27, N(despues)=27
Phi(antes)=-1.3500, Phi(despues)=-1.3500
Termino de shaping = gamma*Phi(despues) - Phi(antes) = 0.0135
N no cambio -- shaping = (gamma-1)*Phi; con gamma<1 y Phi<0 (hay gente activa), (gamma-1)*Phi > 0: un premio chico y constante mientras la cola no se mueva, del orden de (1-gamma)*|Phi|


## Chequeo de sanity para G: la identidad de conservación del tiempo incremental

`G` calcula, paso a paso, `R_t = -sum(deltaT_i)`. Si la fórmula es correcta
y no hay doble conteo ni fugas, la suma de `-R_t` a lo largo de TODO un
episodio debe coincidir EXACTAMENTE con: (tiempo total en sistema de cada
persona que fue entregada durante el episodio) + (tiempo en sistema, hasta
el cierre, de cada persona que quedó activa sin ser entregada). Esta es la
identidad de conservación -- cada minuto que alguien pasa en el sistema se
cuenta una vez, en el paso exacto en que ocurre, sin importar si esa
persona llegó a mitad de un paso o fue entregada a mitad de otro.

In [6]:
env_g = construir_entorno_intercambiable("G")
obs, info = env_g.reset(seed=1001)

suma_incremental = 0.0          # acumula -r de calcular_recompensa_G, paso a paso
tiempo_entrega_exacto = {}      # id de unidad -> minuto EXACTO en que fue entregada (independiente de G)

while True:
    estado = info["_estado_obj"]
    libres = [b for b in estado.barcos if b.libre]
    decisiones = asignar_flota(libres, estado, matriz_tiempos, env_g.capacidad_barco, cfg_sim["recompensa"])
    accion = np.array([env_g.codificar_accion_barco(decisiones[b.id]) if b.libre else 0 for b in estado.barcos])

    len_atendidas_antes = len(env_g.atendidas_historico)
    obs, r, terminated, truncated, info = env_g.step(accion)
    t_este_paso = info["_estado_obj"].t_actual_min
    for u in env_g.atendidas_historico[len_atendidas_antes:]:
        tiempo_entrega_exacto[u.id] = t_este_paso

    suma_incremental += -r
    if truncated or terminated:
        break

t_final = info["_estado_obj"].t_actual_min
activas_al_final = [u for cola in env_g.colas.values() for u in cola] + [u for b in env_g.barcos for u in b.a_bordo]

# Chequeo independiente: tiempo total en sistema de cada persona, usando el minuto de
# entrega EXACTO registrado arriba (no la fórmula de G) para las entregadas, y t_final
# para las que quedaron activas -- no reusa ningún cálculo interno de calcular_recompensa_G.
suma_directa = sum(u.tamano * (tiempo_entrega_exacto[u.id] - u.minuto_llegada) for u in env_g.atendidas_historico)
suma_directa += sum(u.tamano * (t_final - u.minuto_llegada) for u in activas_al_final)

print(f"Suma incremental acumulada (sum de -r de G, paso a paso):        {suma_incremental:.4f}")
print(f"Suma directa (tiempo real en sistema, entrega exacta + activas): {suma_directa:.4f}")
print(f"Diferencia: {suma_incremental - suma_directa:.9f}")
assert abs(suma_incremental - suma_directa) < 1e-6, "la identidad NO se cumple -- revisar calcular_recompensa_G"
print("OK: la identidad se cumple exactamente -- G no tiene doble conteo ni fugas.")


Suma incremental acumulada (sum de -r de G, paso a paso):        2529.5500
Suma directa (tiempo real en sistema, entrega exacta + activas): 2529.5500
Diferencia: 0.000000000
OK: la identidad se cumple exactamente -- G no tiene doble conteo ni fugas.


## Problema de escala en H: ¿el 95/5 representa realmente 95%/5%?

`H = -(peso_tiempo * J_T + peso_movimiento * J_M)`, con `peso_tiempo=0.95`,
`peso_movimiento=0.05`. Esos pesos solo reparten el peso "95%/5%" de forma
correcta si `J_T` y `J_M` están en escalas comparables ANTES de aplicar los
pesos -- si una magnitud es órdenes de magnitud más grande que la otra, el
95/5 nominal no representa una contribución real de 95%/5% a la señal de
recompensa. Se mide directamente, corriendo la política base un episodio
completo con seguimiento de ambos términos SIN ponderar.

In [7]:
env_h = construir_entorno_intercambiable("H")
obs, info = env_h.reset(seed=1001)

suma_j_tiempo_crudo = 0.0
suma_j_movimiento_crudo = 0.0
suma_j_tiempo_ponderado = 0.0
suma_j_movimiento_ponderado = 0.0
n_pasos = 0

while True:
    estado = info["_estado_obj"]
    libres = [b for b in estado.barcos if b.libre]
    decisiones = asignar_flota(libres, estado, matriz_tiempos, env_h.capacidad_barco, cfg_sim["recompensa"])
    accion = np.array([env_h.codificar_accion_barco(decisiones[b.id]) if b.libre else 0 for b in estado.barcos])
    obs, r, terminated, truncated, info = env_h.step(accion)
    d = info["recompensa_desglose"]
    suma_j_tiempo_crudo += d["tiempo_cuadratico"]
    suma_j_movimiento_crudo += d["movimiento"]
    suma_j_tiempo_ponderado += d["tiempo_cuadratico_ponderado"]
    suma_j_movimiento_ponderado += d["movimiento_ponderado"]
    n_pasos += 1
    if truncated or terminated:
        break

print(f"Sobre {n_pasos} pasos (semilla 1001, politica base):\n")
print(f"J_T (tiempo cuadratico) crudo,    acumulado: {suma_j_tiempo_crudo:14.2f}  (promedio/paso: {suma_j_tiempo_crudo/n_pasos:10.4f})")
print(f"J_M (movimiento)        crudo,    acumulado: {suma_j_movimiento_crudo:14.2f}  (promedio/paso: {suma_j_movimiento_crudo/n_pasos:10.4f})")
razon = suma_j_tiempo_crudo / suma_j_movimiento_crudo if suma_j_movimiento_crudo else float("inf")
print(f"\nRazon J_T/J_M (antes de ponderar): {razon:.1f}x")
print(f"\nDespues de aplicar peso_tiempo=0.95 / peso_movimiento=0.05:")
print(f"  contribucion tiempo (95% nominal):      {suma_j_tiempo_ponderado:14.2f}  ({100*suma_j_tiempo_ponderado/(suma_j_tiempo_ponderado+suma_j_movimiento_ponderado):.4f}% real)")
print(f"  contribucion movimiento (5% nominal):   {suma_j_movimiento_ponderado:14.2f}  ({100*suma_j_movimiento_ponderado/(suma_j_tiempo_ponderado+suma_j_movimiento_ponderado):.4f}% real)")


Sobre 90 pasos (semilla 1001, politica base):



J_T (tiempo cuadratico) crudo,    acumulado:      229567.00  (promedio/paso:  2550.7445)
J_M (movimiento)        crudo,    acumulado:          96.00  (promedio/paso:     1.0667)

Razon J_T/J_M (antes de ponderar): 2391.3x

Despues de aplicar peso_tiempo=0.95 / peso_movimiento=0.05:
  contribucion tiempo (95% nominal):           218088.65  (99.9978% real)
  contribucion movimiento (5% nominal):             4.80  (0.0022% real)
